# 11 smote augmentation

Supplementary Random Forest SMOTE experiment. This uses training data only for scaling and SMOTE, returns samples to original units, and compares against the baseline on unchanged validation data. It does not train TabPFN or use a new untouched test set. Synthetic count features can be fractional or unrealistic; no rounding or performance claims are added by this publication cleanup.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


Install dependencies before starting the kernel, using the pinned requirements files.
`python -m pip install -r requirements-smote.txt`
Run that command from the repository root. This notebook does not install or upgrade packages.


In [ ]:

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Project paths
project_root = PUBLICATION_ROOT

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data/notebook_outputs/ml_ready"

# Load the original training and validation datasets
train_df = pd.read_csv(data_dir / "train_ml.csv")
val_df = pd.read_csv(data_dir / "validation_ml.csv")

# Load the existing Random Forest baseline
rf_baseline = joblib.load(
    project_root / "models/random_forest_baseline.joblib"
)

# Preserve the original model's feature order
feature_cols = (
    list(rf_baseline.feature_names_in_)
    if hasattr(rf_baseline, "feature_names_in_")
    else [
        col for col in train_df.columns
        if col not in ["SessionID", "GroundTruthLabel"]
    ]
)

X_train = train_df[feature_cols]
y_train = train_df["GroundTruthLabel"].astype(int)

X_val = val_df[feature_cols]
y_val = val_df["GroundTruthLabel"].astype(int)

print("ORIGINAL TRAINING DISTRIBUTION")
print(Counter(y_train))

# Standardise training features for SMOTE's
# nearest-neighbour calculations
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Apply SMOTE to the TRAINING DATA ONLY
smote = SMOTE(
    sampling_strategy="auto",
    k_neighbors=5,
    random_state=42
)

X_smote_scaled, y_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

# Return features to their original numeric scale
# so Random Forest receives the original feature units
X_smote = pd.DataFrame(
    scaler.inverse_transform(X_smote_scaled),
    columns=feature_cols
)

print("\nAFTER SMOTE")
print(Counter(y_smote))
print("Synthetic sessions added:",
      len(y_smote) - len(y_train))

# Train one temporary Random Forest
rf_smote = clone(rf_baseline)
rf_smote.fit(X_smote, y_smote)

# Compare both models on unchanged validation data
baseline_predictions = rf_baseline.predict(X_val)
smote_predictions = rf_smote.predict(X_val)

def evaluate_model(name, predictions):
    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    return {
        "Model": name,
        "Accuracy": round(
            accuracy_score(y_val, predictions), 4
        ),
        "Precision": round(
            precision_score(y_val, predictions, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(y_val, predictions, zero_division=0), 4
        ),
        "F1": round(
            f1_score(y_val, predictions, zero_division=0), 4
        ),
        "False positives": int(fp),
        "False negatives": int(fn)
    }

comparison_df = pd.DataFrame([
    evaluate_model("Original Random Forest",
                   baseline_predictions),
    evaluate_model("Random Forest + SMOTE",
                   smote_predictions)
])

print("\nVALIDATION PERFORMANCE COMPARISON")
display(comparison_df)


In [ ]:

import numpy as np
import pandas as pd

# Number of original and synthetic samples
n_original = len(train_df)
n_synthetic = len(X_smote) - n_original

# Preserve the original training rows exactly
original_rows = train_df[
    ["SessionID"] + feature_cols + ["GroundTruthLabel"]
].copy()

original_rows["SampleOrigin"] = "Original"

# Extract only the newly generated SMOTE samples
synthetic_rows = X_smote.iloc[n_original:].copy()
synthetic_rows = synthetic_rows[feature_cols].reset_index(drop=True)

# Assign unique identifiers to synthetic samples
synthetic_rows.insert(
    0,
    "SessionID",
    [
        f"SMOTE_SYN_{i:04d}"
        for i in range(1, n_synthetic + 1)
    ]
)

synthetic_rows["GroundTruthLabel"] = (
    np.asarray(y_smote)[n_original:].astype(int)
)

synthetic_rows["SampleOrigin"] = "SMOTE Synthetic"

# Combine original and synthetic training samples
smote_training_df = pd.concat(
    [original_rows, synthetic_rows],
    ignore_index=True
)

# Save as a NEW dataset
# The original train_ml.csv remains untouched
smote_file = data_dir / "train_ml_smote.csv"

smote_training_df.to_csv(smote_file, index=False)

print("SMOTE DATASET EXPORTED SUCCESSFULLY")
print("-" * 50)

print("Original training sessions:", n_original)
print("Synthetic SMOTE samples:", n_synthetic)
print("Total augmented samples:", len(smote_training_df))

print("\nCLASS DISTRIBUTION")
display(
    pd.crosstab(
        smote_training_df["SampleOrigin"],
        smote_training_df["GroundTruthLabel"],
        margins=True
    )
)

print("\nSaved to:")
print(smote_file)

display(smote_training_df.head())
